Auto Annotation using scArches

adapted from: https://docs.scarches.org/en/latest/hlca_map_classify.html

In [1]:
import os
import sys
import scanpy as sc
import numpy as np
import pandas as pd
import scarches as sca
import anndata as ad
from scipy import sparse
import gdown
import gzip
import shutil
import urllib.request
from pathlib import Path

 captum (see https://github.com/pytorch/captum).


In [2]:
#####
# Setting Directories
#####

# directory in which to store the reference model directory
REF_MODEL_DIR = Path.home()/"project/crc-pm/data/Marteau_2026_CC/crc_atlas_models_minified/crc_atlas_scanvi_model"

# directory in which to store result model of the query
SURGERY_MODEL_DIR = Path.home()/'project/crc-pm/result/query_model/' 

# directory in which to store the query
PATH_QUERY_DATA = Path.home()/"project/crc-pm/result/qc/qc.h5ad"

In [3]:
adata_query_unprep = sc.read_h5ad(PATH_QUERY_DATA)

In [ ]:
# padding
adata_query = sca.models.SCANVI.prepare_query_anndata(
    adata=adata_query_unprep, reference_model=str(REF_MODEL_DIR), inplace=False
)
# Prepare obs
adata_query.obs['dataset'] = adata_query_unprep.obs['Study']
adata_query.obs['batch'] = adata_query_unprep.obs['Batch']
adata_query.obs["cell_type"] = "Unknown"

INFO     File                                                                                                      
         /home/zhekai/project/crc-pm/data/Marteau_2026_CC/crc_atlas_models_minified/crc_atlas_scanvi_model/model.pt
         already downloaded                                                                                        
INFO     Found 98.55903935957305% reference vars in query data.                                                    


In [ ]:
surgery_model = sca.models.SCANVI.load_query_data(
    adata_query,
    str(REF_MODEL_DIR),
    freeze_dropout=True,
)

INFO     File                                                                                                      
         /home/zhekai/project/crc-pm/data/Marteau_2026_CC/crc_atlas_models_minified/crc_atlas_scanvi_model/model.pt
         already downloaded                                                                                        


In [ ]:
# parameter settings copied directly from tutorial
# dk what they mean
surgery_epochs = 500
early_stopping_kwargs_surgery = {
    "early_stopping": True,
    "early_stopping_monitor": "elbo_validation",
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001,
    "plan_kwargs": {"weight_decay": 0.0},
}

In [ ]:
surgery_model.registry_["setup_args"]

{'labels_key': 'cell_type',
 'unlabeled_category': 'Unknown',
 'layer': None,
 'batch_key': 'dataset',
 'size_factor_key': None,
 'categorical_covariate_keys': ['batch'],
 'continuous_covariate_keys': None,
 'use_minified': False}

In [ ]:
#####
# Train model
#####
surgery_model.train(max_epochs=surgery_epochs, **early_stopping_kwargs_surgery)
# save model
surgery_model.save(str(SURGERY_MODEL_DIR), overwrite=True)

INFO     Training for 500 epochs.                                                                                  


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for perfo

Training:   0%|          | 0/500 [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


Exception raised during training. <class 'NameError'> 1


SystemExit: 1

/home/zhekai/miniconda3/envs/scarches/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


---

In [ ]:
# # read
# surgery_model = sca.models.SCANVI.load(SURGERY_MODEL_DIR, adata_query)
# get latent representation of query
adata_query_latent = sc.AnnData(surgery_model.get_latent_representation(adata_query))
# copy obs
adata_query_latent.obs = adata_query.obs.copy()
# read in ref embedding and get latent (!!! X_latent is the result of scVI, not scANVI)
adata_atlas = sc.read_h5ad(REF_MODEL_DIR/'adata.h5ad')
adata_ref_latent = sc.AnnData(
    X=adata_atlas.obsm['scanvi_latent_qzm'].copy(),
    obs=adata_atlas.obs.copy()
)

INFO     File /home/zhekai/project/crc-pm/result/query_model/model.pt already downloaded                           


In [ ]:
# combine ref and query
adata_query_latent.obs["ref_or_query"] = "query"
adata_ref_latent.obs["ref_or_query"] = "ref"
# del adata_ref_latent.obs["cell_type_predicted"]
combined_emb = sc.concat(
    (adata_ref_latent, adata_query_latent), join="outer",
)
combined_emb.obs.index.is_unique

True

In [ ]:
# label Transfer
knn_transformer = sca.utils.knn.weighted_knn_trainer(
    train_adata=adata_ref_latent,
    train_adata_emb="X",  # location of our joint embedding
    n_neighbors=50,
)

Weighted KNN with n_neighbors = 50 ... 

In [ ]:
labels, uncert = sca.utils.knn.weighted_knn_transfer(
    query_adata=adata_query_latent,
    query_adata_emb="X", # location of our embedding, query_adata.X in this case
    label_keys="cell_type", # obs column name(s) for which to transfer labels
    knn_model=knn_transformer,
    ref_adata_obs=adata_ref_latent.obs[["cell_type"]],
    # note that adata_ref_latent has cell_type and cell_type_predicted, only send one
    # the cell_type_predicted in ref is generated by re-appling classifier to data
)

/home/zhekai/miniconda3/envs/scarches/lib/python3.11/site-packages/scarches/utils/knn.py:130: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  uncertainties.iloc[i][j] = (max(1 - best_prob, 0))
/home/zhekai/miniconda3/envs/scarches/lib/python3.

finished!


In [ ]:
labels.rename(columns={"cell_type": "cell_type_transferred_label_unfiltered"},inplace=True,)
uncert.rename(columns={"cell_type": "cell_type_transfer_uncert"},inplace=True,)
combined_emb.obs = combined_emb.obs.rename(columns={'cell_type_predicted':'orig_cell_type_predicted'})
combined_emb.obs = combined_emb.obs.join(labels)
combined_emb.obs = combined_emb.obs.join(uncert)
# transfer data type to float
combined_emb.obs["cell_type_transfer_uncert"].dtype
combined_emb.obs["cell_type_transfer_uncert"] = pd.to_numeric(
    combined_emb.obs["cell_type_transfer_uncert"],
    errors="raise",
)

---

In [ ]:
transferred_labels = combined_emb.obs['cell_type_transferred_label_unfiltered'].astype('category')
if 'Unknown' not in transferred_labels.cat.categories:
    transferred_labels = transferred_labels.cat.add_categories(['Unknown'])
combined_emb.obs['cell_type_transferred_label_unfiltered'] = transferred_labels

uncertainty_threshold = 0.2
combined_emb.obs['cell_type_transferred_label_filtered'] = combined_emb.obs['cell_type_transferred_label_unfiltered'].mask(
    combined_emb.obs['cell_type_transfer_uncert'] > uncertainty_threshold, 'Unknown')

query_mask = combined_emb.obs['ref_or_query'].eq('query')
unknown_pct = combined_emb.obs.loc[
    query_mask, 'cell_type_transferred_label_filtered'
].eq('Unknown').mean() * 100
print(np.round(unknown_pct, 2))

39.59


In [ ]:
combined_emb.obs.loc[
    combined_emb.obs['ref_or_query'] == 'query',
    'cell_type_transfer_uncert'
].describe()

count    340545.000000
mean          0.190994
std           0.185850
min           0.000000
25%           0.020000
50%           0.139652
75%           0.339696
max           0.879949
Name: cell_type_transfer_uncert, dtype: float64

In [ ]:
sc.pp.neighbors(combined_emb, n_neighbors=30)
sc.tl.umap(combined_emb)
sc.pl.umap(combined_emb, color="ref_or_query", frameon=False, wspace=0.6)
sc.pl.umap(
    combined_emb,
    color='cell_type_transfer_uncert',
    frameon=False,
)

In [ ]:
query_mask = combined_emb.obs['ref_or_query'].eq('query')
query_obs = combined_emb.obs.loc[query_mask].reindex(adata_query_unprep.obs_names)

if query_obs.index.has_duplicates:
    raise ValueError('Query cell IDs are not unique')
if query_obs['cell_type_transferred_label_filtered'].isna().any():
    raise ValueError('Some query cells did not receive transferred labels')

adata_query_unprep.obs['scANVI_CellType'] = query_obs[
    'cell_type_transferred_label_filtered'
].array
adata_query_unprep.obs['scANVI_CellType_uncertainty'] = query_obs[
    'cell_type_transfer_uncert'
].to_numpy()

OUTPUT_DIR = Path.home() / 'project/crc-pm/result/label-transfer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete reference + query embedding as its own AnnData object.
# This preserves X, obs, UMAP coordinates, and the neighbor graph together.
combined_emb.write_h5ad(OUTPUT_DIR / 'combined_embedding.h5ad')
adata_query_unprep.write_h5ad(OUTPUT_DIR / 'qc_and_transferred.h5ad')